# Run the W&B Sweep on a Colab GPU

A **sweep** is a search over training settings that W&B coordinates. The grid lives in `sweeps/grid.yaml` in the repo: 3 encoders × 3 hard-negative strategies × strength normalizer on/off = 18 runs. This notebook starts an **agent**: a loop that asks W&B for the next untried combination, trains it with the same `train()` as before, reports `val/acc@1`, and repeats.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Colab Secret `WANDB_API_KEY` (same as notebook 01).
3. Register the sweep once, locally: `uv run scripts/08_sweep.py --create`. It prints a sweep id like `kettle-labs/rxnorm-vandf/ab12cd34`. Paste it in cell 5.

Expect about 75 minutes for all 18 runs on a T4 (MiniLM ~1.5 min each, bge-small ~3, SapBERT ~8). The sweep page updates live. Background: `docs/calibration-and-sweeps.md`.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Get the code
Same as notebook 01: a plain clone of the public repo.

In [ ]:
import os, subprocess

REPO = "kvenanzi/rxnorm"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("rxnorm"):
    subprocess.run(["git", "clone", "-q", URL], check=True)
else:
    subprocess.run(["git", "-C", "rxnorm", "pull", "-q"], check=True)
!git -C rxnorm log --oneline -1

## 3. Make the package importable and install what Colab lacks

In [ ]:
import sys
if os.path.abspath("rxnorm") not in sys.path:
    sys.path.insert(0, os.path.abspath("rxnorm"))
%pip install -q duckdb sentence-transformers datasets wandb accelerate
import torch, sentence_transformers, rxnorm_vandf
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| sentence-transformers", sentence_transformers.__version__,
      "| rxnorm_vandf from", os.path.dirname(rxnorm_vandf.__file__))

## 4. Log in to Weights & Biases

In [ ]:
import wandb
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 5. Start the agent
`run_one` is what the agent calls for each trial. It builds a `TrainConfig` with the fixed choices (pull data from the artifact, don't log a model per trial) and passes `sweep=True`, which tells `train()` to take this trial's parameters from `wandb.config`.

`count=18` runs the whole grid. If the session disconnects partway, just rerun this cell: W&B hands out only the combinations that haven't been done. You can also start a second agent on your local GPU with `uv run scripts/08_sweep.py --agent <id>`; the two share the grid.

In [ ]:
import gc
from rxnorm_vandf.train import TrainConfig, train

SWEEP_ID = "kettle-labs/rxnorm-vandf/PASTE-SWEEP-ID-HERE"

def run_one():
    train(TrainConfig(data_dir=None, output_dir="models", log_model=False,
                      tags=["sweep", "colab"]), sweep=True)
    gc.collect(); torch.cuda.empty_cache()

wandb.agent(SWEEP_ID, function=run_one, count=18)

## 6. Reading the sweep page
Open the sweep from the **Sweeps** entry in the project's left sidebar.
- **Parallel coordinates plot:** one line per run through the parameter axes, colored by `val/acc@1`. Look for which axis separates high from low lines.
- **Parameter importance:** W&B's estimate of which setting mattered most (correlation with the metric).
- **Runs table:** group by `base_model` or `negatives` (the *Group* button) to compare averages.

Pick the best `val/acc@1` (not test: test is for the final number), retrain that config with `log_model=True`, and calibrate it with `scripts/07_calibrate.py`.